In [1]:
!pip install pyspark

In [3]:
from google.colab import files

uploaded = files.upload()

Saving plant_master.csv to plant_master.csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv
Saving machine_master.csv to machine_master.csv
Saving machine_configurations.json to machine_configurations.json


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ManufacturingProduction")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("PySpark started successfully!")

PySpark started successfully!


In [6]:
df = spark.read.csv(
    "/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

extra_df = spark.read.csv(
    "/content/manufacturing_production_extra_12k.csv",
    header=True,
    inferSchema=True
)

machine_df = spark.read.csv(
    "/content/machine_master.csv",
    header=True,
    inferSchema=True
)

plant_df = spark.read.csv(
    "/content/plant_master.csv",
    header=True,
    inferSchema=True
)

machine_json = (
    spark.read
    .option("multiline", "true")
    .json("/content/machine_configurations.json")
)

print("All five files loaded successfully!")

All five files loaded successfully!


In [7]:
print("Main production:", df.count())
print("Extra production:", extra_df.count())
print("Machine master:", machine_df.count())
print("Plant master:", plant_df.count())
print("Machine JSON:", machine_json.count())

Main production: 120000
Extra production: 12500
Machine master: 30
Plant master: 6
Machine JSON: 30


**Load the Data in PySpark**

In [8]:
df.show(20, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No              |Charan    |
|PR0000002    |2026-

In [9]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [10]:
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [14]:
print("Total records:", df.count())

Total records: 120000


In [12]:
df.select(
    "production_id",
    "plant_id",
    "machine_id",
    "product_family",
    "units_produced"
).show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [13]:
df.dtypes

[('production_id', 'string'),
 ('production_date', 'date'),
 ('plant_id', 'string'),
 ('machine_id', 'string'),
 ('operator_id', 'string'),
 ('shift', 'string'),
 ('product_code', 'string'),
 ('product_family', 'string'),
 ('units_planned', 'int'),
 ('units_produced', 'int'),
 ('defective_units', 'int'),
 ('downtime_minutes', 'int'),
 ('energy_kwh', 'double'),
 ('temperature_c', 'double'),
 ('vibration_mm_s', 'double'),
 ('quality_status', 'string'),
 ('maintenance_flag', 'string'),
 ('supervisor', 'string')]

In [15]:
df.sample(
    withReplacement=False,
    fraction=0.001,
    seed=42
).show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0007283    |2026-03-25     |PL05    |MC025     |OP0320     |Night  |MTR-100     |Motors        |815          |676           |22             |10              |259.99    |47.16        |3.76          |Pass          |No              |Charan    |
|PR0007396    |2026-

**Creating DataFrames**

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import lit

spark = SparkSession.builder.appName("PlantDataFrame").getOrCreate()

In [17]:
data = [
    (101, "Chennai Plant", "Chennai"),
    (102, "Bangalore Plant", "Bangalore"),
    (103, "Hyderabad Plant", "Hyderabad"),
    (104, "Mumbai Plant", "Mumbai")
]

In [18]:
columns = ["plant_id", "plant_name", "city"]

plant_df = spark.createDataFrame(data, columns)

plant_df.show()

+--------+---------------+---------+
|plant_id|     plant_name|     city|
+--------+---------------+---------+
|     101|  Chennai Plant|  Chennai|
|     102|Bangalore Plant|Bangalore|
|     103|Hyderabad Plant|Hyderabad|
|     104|   Mumbai Plant|   Mumbai|
+--------+---------------+---------+



In [19]:
schema = StructType([
    StructField("plant_id", IntegerType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

plant_df_schema = spark.createDataFrame(data, schema)

plant_df_schema.show()

+--------+---------------+---------+
|plant_id|     plant_name|     city|
+--------+---------------+---------+
|     101|  Chennai Plant|  Chennai|
|     102|Bangalore Plant|Bangalore|
|     103|Hyderabad Plant|Hyderabad|
|     104|   Mumbai Plant|   Mumbai|
+--------+---------------+---------+



In [20]:
plant_df_schema.printSchema()

root
 |-- plant_id: integer (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [21]:
plant_df_schema = plant_df_schema.withColumn(
    "country",
    lit("India")
)

plant_df_schema.show()

+--------+---------------+---------+-------+
|plant_id|     plant_name|     city|country|
+--------+---------------+---------+-------+
|     101|  Chennai Plant|  Chennai|  India|
|     102|Bangalore Plant|Bangalore|  India|
|     103|Hyderabad Plant|Hyderabad|  India|
|     104|   Mumbai Plant|   Mumbai|  India|
+--------+---------------+---------+-------+



In [22]:
from pyspark.sql.types import *
from pyspark.sql.functions import lit

data = [
    (101, "Chennai Plant", "Chennai"),
    (102, "Bangalore Plant", "Bangalore"),
    (103, "Hyderabad Plant", "Hyderabad"),
    (104, "Mumbai Plant", "Mumbai")
]

schema = StructType([
    StructField("plant_id", IntegerType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

plant_df = spark.createDataFrame(data, schema)

plant_df.printSchema()

plant_df = plant_df.withColumn("country", lit("India"))

plant_df.show()

root
 |-- plant_id: integer (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)

+--------+---------------+---------+-------+
|plant_id|     plant_name|     city|country|
+--------+---------------+---------+-------+
|     101|  Chennai Plant|  Chennai|  India|
|     102|Bangalore Plant|Bangalore|  India|
|     103|Hyderabad Plant|Hyderabad|  India|
|     104|   Mumbai Plant|   Mumbai|  India|
+--------+---------------+---------+-------+



**Pandas DataFrame**

In [23]:
import pandas as pd

pdf = pd.read_csv("/content/manufacturing_production_120k.csv")

In [24]:
from google.colab import files

uploaded = files.upload()

Saving manufacturing_production_120k.csv to manufacturing_production_120k (1).csv


In [25]:
pdf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [26]:
pdf.tail()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
119995,PR0119996,2026-07-13,PL05,MC022,OP0193,Evening,MTR-200,Motors,435,399,1,73.0,126.07,62.79,5.73,Rework,No,Bharat
119996,PR0119997,2026-04-26,PL05,MC024,OP0124,Evening,CTL-130,Controllers,813,777,8,49.0,318.22,63.05,3.53,Pass,No,Irfan
119997,PR0119998,2026-02-01,PL06,MC027,OP0051,Evening,VLV-120,Valves,650,597,13,0.0,269.46,60.52,3.81,Pass,No,Bharat
119998,PR0119999,2026-01-17,PL01,MC001,OP0452,Evening,MTR-300,Motors,887,899,22,21.0,184.62,64.89,5.20,Pass,Yes,Farah
119999,PR0120000,2026-08-16,PL02,MC007,OP0420,Morning,MTR-200,Motors,547,477,4,43.0,107.95,62.50,3.97,Pass,No,Farah


In [27]:
print(pdf.shape)

(120000, 18)


In [28]:
print(pdf.columns)

Index(['production_id', 'production_date', 'plant_id', 'machine_id',
       'operator_id', 'shift', 'product_code', 'product_family',
       'units_planned', 'units_produced', 'defective_units',
       'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s',
       'quality_status', 'maintenance_flag', 'supervisor'],
      dtype='object')


In [29]:
print(pdf.dtypes)

production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


In [30]:
pdf[["plant_id", "machine_id", "units_produced"]]

,plant_id,machine_id,units_produced
0,PL03,MC011,993
1,PL04,MC019,836
2,PL01,MC002,380
3,PL03,MC013,720
4,PL04,MC020,296
...,...,...,...
119995,PL05,MC022,399
119996,PL05,MC024,777
119997,PL06,MC027,597
119998,PL01,MC001,899


In [31]:
pdf = pdf.rename(columns={
    "units_produced": "production_output"
})

pdf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [32]:
rejected_df = pdf[pdf["quality_status"] == "Reject"]

rejected_df.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862,895,51,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532,527,26,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623,560,23,14.0,217.69,55.23,3.28,Reject,Yes,Farah


In [33]:
print("Rejected records:", len(rejected_df))

Rejected records: 22744


In [36]:
high_temp_df = pdf[pdf["temperature_c"] > 75]

high_temp_df.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
14,PR0000015,2026-03-15,PL02,MC007,OP0238,Morning,MTR-300,Motors,1086,1042,4,172.0,239.64,76.39,2.81,Pass,No,Deepa
16,PR0000017,2026-08-04,PL03,MC011,OP0533,Morning,CTL-330,Controllers,1152,963,28,91.0,347.34,75.09,4.00,Pass,Yes,Gopal
19,PR0000020,2026-02-18,PL03,MC014,OP0563,Morning,PMP-310,Pumps,646,573,13,64.0,277.73,76.92,2.35,Pass,No,Irfan
26,PR0000027,2026-01-03,PL05,MC023,OP0190,Morning,PMP-210,Pumps,358,310,6,46.0,78.29,75.42,3.72,Rework,No,Anita
54,PR0000055,2026-07-31,PL01,MC005,OP0212,Morning,CTL-230,Controllers,499,478,10,32.0,112.60,75.10,3.38,Pass,No,Farah


In [37]:
print("High temperature records:", len(high_temp_df))

High temperature records: 6505


In [42]:
plant_pdf = pd.read_csv("/content/plant_master.csv")

merged_df = pdf.merge(
    plant_pdf[["plant_id", "city"]],
    on="plant_id",
    how="left"
)

night_hyderabad_df = merged_df[
    (merged_df["shift"] == "Night") &
    (merged_df["city"] == "Hyderabad")
]

night_hyderabad_df.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,production_output,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor,city
29,PR0000030,2026-09-18,PL01,MC004,OP0266,Night,VLV-320,Valves,358,359,1,19.0,107.98,72.00,1.73,Pass,No,Bharat,Hyderabad
38,PR0000039,2026-07-17,PL01,MC002,OP0220,Night,VLV-120,Valves,260,265,0,2.0,127.18,51.87,3.41,Pass,No,Irfan,Hyderabad
62,PR0000063,2026-02-18,PL01,MC002,OP0390,Night,VLV-120,Valves,656,665,0,44.0,311.80,58.34,2.00,Pass,No,Deepa,Hyderabad
79,PR0000080,2026-06-10,PL01,MC003,OP0214,Night,CTL-130,Controllers,948,821,49,20.0,188.65,62.68,2.85,Reject,No,Anita,Hyderabad
145,PR0000146,2026-08-19,PL01,MC005,OP0088,Night,VLV-120,Valves,1156,969,0,9.0,501.65,67.41,5.76,Rework,No,Bharat,Hyderabad


**Dask DataFrame**

In [43]:
!pip install "dask[dataframe]"

In [44]:
import dask.dataframe as dd

ddf = dd.read_csv("/content/manufacturing_production_120k.csv")

In [48]:
ddf = dd.read_csv("/content/manufacturing_production_120k.csv", dtype={'downtime_minutes': 'float64'})
ddf.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0000001,2026-08-07,PL03,MC011,OP0275,Evening,MTR-100,Motors,1060,993,30,34.0,388.11,61.40,3.88,Rework,No,Charan
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
3,PR0000004,2026-01-15,PL03,MC013,OP0076,Morning,PMP-210,Pumps,787,720,7,39.0,217.61,64.47,NaN,Rework,No,Farah
4,PR0000005,2026-09-21,PL04,MC020,OP0501,Morning,PMP-110,Pumps,344,296,7,49.0,115.47,72.59,3.21,Pass,No,Deepa


In [49]:
print("Number of partitions:", ddf.npartitions)

Number of partitions: 1


In [50]:
ddf[["plant_id", "product_family", "units_produced"]].head()

,plant_id,product_family,units_produced
0,PL03,Motors,993
1,PL04,Controllers,836
2,PL01,Motors,380
3,PL03,Pumps,720
4,PL04,Pumps,296


In [51]:
rejected_df = ddf[ddf["quality_status"] == "Reject"]

rejected_df.head()

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862,895,51,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532,527,26,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623,560,23,14.0,217.69,55.23,3.28,Reject,Yes,Farah


In [52]:
print(rejected_df.shape[0].compute())

22744


In [53]:
avg_production = ddf.groupby("plant_id")["units_produced"].mean()

avg_production.compute()

,units_produced
plant_id,
PL01,672.998304
PL02,672.781326
PL03,675.665503
PL04,673.835101
PL05,673.124994
PL06,678.006884


In [54]:
total_defects = ddf.groupby("product_family")["defective_units"].sum()

total_defects.compute()

,defective_units
product_family,
Controllers,507781
Motors,505287
Pumps,507864
Valves,509312


In [55]:
import pandas as pd

pdf = pd.read_csv("/content/manufacturing_production_120k.csv")

pandas_result = pdf.groupby("plant_id")["units_produced"].mean()

print(pandas_result)

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


In [56]:
dask_result = ddf.groupby("plant_id")["units_produced"].mean().compute()

print(dask_result)

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


**Manipulating DataFrames**

In [68]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [69]:
df = df.withColumn(
    "production_efficiency",
    (F.col("units_produced") / F.col("units_planned")) * 100
)

In [70]:
df = df.withColumn(
    "good_units",
    F.col("units_produced") - F.col("defective_units")
)

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    3

In [63]:
df = df.withColumn("country", F.lit("India")).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993

In [71]:
df = df.withColumnRenamed("energy_kwh", "energy_consumed")
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|       

In [73]:
df = df.withColumn(
    "units_produced",
    F.col("units_produced").cast("double")
)
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|         993.0|             30|       

In [75]:
df = df.withColumn(
    "defect_percentage",
    F.when(
        F.col("units_produced") != 0,
        (F.col("defective_units") / F.col("units_produced")) * 100
    ).otherwise(0)
)
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units| defect_percentage|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Mo

In [76]:
df = df.withColumn(
    "energy_per_unit",
    F.when(
        F.col("units_produced") != 0,
        F.col("energy_consumed") / F.col("units_produced")
    ).otherwise(0)
)
df.show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+------------------+-------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units| defect_percentage|    energy_per_unit|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+---------------------+----------+------------------+-------------------+
|    PR0000001|     2026-08-07|   

In [78]:
from pyspark.sql import functions as F
df = df.withColumn("country", F.lit("India"))

df.select(
    "production_efficiency",
    "good_units",
    "country",
    "energy_consumed",
    "units_produced",
    "defect_percentage",
    "energy_per_unit"
).show(20, truncate=False)

+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|production_efficiency|good_units|country|energy_consumed|units_produced|defect_percentage |energy_per_unit    |
+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|93.67924528301886    |963       |India  |388.11         |993.0         |3.0211480362537766|0.39084592145015107|
|103.33745364647713   |836       |India  |398.58         |836.0         |0.0               |0.4767703349282296 |
|85.97285067873304    |363       |India  |183.34         |380.0         |4.473684210526316 |0.48247368421052633|
|91.4866581956798     |713       |India  |217.61         |720.0         |0.9722222222222222|0.30223611111111115|
|86.04651162790698    |289       |India  |115.47         |296.0         |2.364864864864865 |0.3901013513513513 |
|82.98538622129436    |745       |India  |290.26         |795.0         |6.289308176100629 |0.36

**Dropping Data**


In [79]:
df = df.drop("supervisor")

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+---------------------+----------+------------------+-------------------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|production_efficiency|good_units| defect_percentage|    energy_per_unit|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+---------------------+----------+------------------+-------------------+-------+
|    PR0000001|     2026-08-07|    PL03|   

In [80]:
df = df.drop("supervisor", "maintenance_flag")

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|production_efficiency|good_units| defect_percentage|    energy_per_unit|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Mo

In [81]:
df = df.dropDuplicates()

print("Records after removing duplicates:", df.count())

Records after removing duplicates: 120000


In [82]:
df = df.dropDuplicates(["production_id"])

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|production_efficiency|good_units| defect_percentage|    energy_per_unit|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Control

In [83]:
df = df.filter(
    df["operator_id"].isNotNull()
)

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|production_efficiency|good_units| defect_percentage|    energy_per_unit|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|    PR0000002|     2026-08-03|    PL04|     MC019|     OP0489|Morning|     CTL-330|   Control

In [86]:
df = df.filter(
    (
        df["temperature_c"].isNull() &
        df["vibration_mm_s"].isNull()
    )
)

df.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|production_efficiency|good_units| defect_percentage|    energy_per_unit|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+---------------------+----------+------------------+-------------------+-------+
|    PR0009219|     2026-03-01|    PL03|     MC015|     OP0062|  Night|     PMP-110|         P

In [87]:
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



In [88]:
print(
    "Unique product families:",
    df.select("product_family").distinct().count()
)

Unique product families: 4


In [89]:
df.select(
    "plant_id",
    "machine_id"
).distinct().show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL04|     MC019|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL04|     MC020|
|    PL01|     MC005|
|    PL04|     MC016|
|    PL01|     MC002|
|    PL03|     MC015|
|    PL03|     MC014|
|    PL02|     MC008|
|    PL05|     MC023|
|    PL01|     MC004|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL05|     MC021|
|    PL02|     MC006|
|    PL03|     MC011|
|    PL05|     MC024|
+--------+----------+
only showing top 20 rows


In [90]:
print(
    "Unique plant-machine combinations:",
    df.select("plant_id", "machine_id").distinct().count()
)

Unique plant-machine combinations: 21


**Sorting**

In [91]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [92]:
df.orderBy("units_produced", ascending=True).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [93]:
df.orderBy("units_produced", ascending=False).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [94]:
df.orderBy("defective_units", ascending=False).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [95]:
df.orderBy(
    F.col("plant_id").asc(),
    F.col("units_produced").desc()
).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [96]:
df.orderBy(
    F.col("units_produced").desc()
).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [97]:
df.orderBy(
    F.col("downtime_minutes").desc()
).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [99]:
df.select(
    "machine_id",
    "plant_id",
    "vibration_mm_s"
).orderBy(
    F.col("vibration_mm_s").desc_nulls_last()
).show(20)

+----------+--------+--------------+
|machine_id|plant_id|vibration_mm_s|
+----------+--------+--------------+
|     MC012|    PL03|          8.56|
|     MC030|    PL06|          8.25|
|     MC027|    PL06|          7.85|
|     MC010|    PL02|           7.8|
|     MC018|    PL04|          7.79|
|     MC024|    PL05|          7.77|
|     MC015|    PL03|          7.76|
|     MC027|    PL06|          7.75|
|     MC015|    PL03|          7.67|
|     MC016|    PL04|          7.63|
|     MC007|    PL02|          7.61|
|     MC010|    PL02|          7.59|
|     MC027|    PL06|          7.58|
|     MC017|    PL04|          7.52|
|     MC021|    PL05|           7.5|
|     MC021|    PL05|           7.5|
|     MC004|    PL01|          7.48|
|     MC027|    PL06|          7.47|
|     MC025|    PL05|          7.46|
|     MC019|    PL04|          7.45|
+----------+--------+--------------+
only showing top 20 rows


In [104]:
df.groupBy("machine_id").agg(
    F.max("vibration_mm_s").alias("highest_vibration")
).orderBy(
    F.col("highest_vibration").desc_nulls_last()
).show()

+----------+-----------------+
|machine_id|highest_vibration|
+----------+-----------------+
|     MC012|             8.56|
|     MC030|             8.25|
|     MC027|             7.85|
|     MC010|              7.8|
|     MC018|             7.79|
|     MC024|             7.77|
|     MC015|             7.76|
|     MC016|             7.63|
|     MC007|             7.61|
|     MC017|             7.52|
|     MC021|              7.5|
|     MC004|             7.48|
|     MC025|             7.46|
|     MC019|             7.45|
|     MC028|             7.43|
|     MC022|              7.4|
|     MC014|             7.37|
|     MC006|             7.36|
|     MC008|             7.32|
|     MC023|             7.29|
+----------+-----------------+
only showing top 20 rows


**Aggregations**

In [105]:
from pyspark.sql import functions as F

In [106]:
df.agg(
    F.sum("units_planned").alias("total_units_planned")
).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+



In [107]:
df.agg(
    F.sum("units_produced").alias("total_units_produced")
).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



In [108]:
df.agg(
    F.sum("defective_units").alias("total_defective_units")
).show()

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+



In [109]:
df.agg(
    F.avg("units_produced").alias("average_units_produced")
).show()

+----------------------+
|average_units_produced|
+----------------------+
|     674.3990666666666|
+----------------------+



In [110]:
df.agg(
    F.max("units_produced").alias("maximum_production")
).show()

+------------------+
|maximum_production|
+------------------+
|              1245|
+------------------+



In [111]:
df.agg(
    F.min("units_produced").alias("minimum_production")
).show()

+------------------+
|minimum_production|
+------------------+
|               205|
+------------------+



In [112]:
df.agg(
    F.avg("downtime_minutes").alias("average_downtime")
).show()

+-----------------+
| average_downtime|
+-----------------+
|33.93149474428351|
+-----------------+



In [113]:
df.agg(
    F.sum("energy_kwh").alias("total_energy_consumed")
).show()

+---------------------+
|total_energy_consumed|
+---------------------+
| 2.8368014080000117E7|
+---------------------+



In [117]:
df.agg(
    F.avg("temperature_c").alias("average_temperature")
).show()

+-------------------+
|average_temperature|
+-------------------+
| 63.021532392167266|
+-------------------+



In [120]:
df.agg(
    F.avg("vibration_mm_s").alias("average_vibration")
).show()

+-----------------+
|average_vibration|
+-----------------+
|3.398737691604917|
+-----------------+




**groupBy()**

In [122]:
from pyspark.sql import functions as F

In [123]:
df.groupBy("plant_id").agg(
    F.sum("units_produced").alias("total_production")
).orderBy(
    F.col("total_production").desc()
).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



In [124]:
df.groupBy("plant_id").agg(
    F.sum("defective_units").alias("total_defects")
).orderBy(
    F.col("total_defects").desc()
).show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



In [125]:
df.groupBy("shift").agg(
    F.avg("downtime_minutes").alias("average_downtime")
).orderBy(
    F.col("average_downtime").desc()
).show()

+-------+------------------+
|  shift|  average_downtime|
+-------+------------------+
|Evening|  34.2213017151833|
|  Night|33.953058820580935|
|Morning| 33.61973080530278|
+-------+------------------+



In [126]:
df.groupBy("product_family").agg(
    F.avg("energy_kwh").alias("average_energy_usage")
).orderBy(
    F.col("average_energy_usage").desc()
).show()

+--------------+--------------------+
|product_family|average_energy_usage|
+--------------+--------------------+
|        Motors|  237.21818224597516|
|         Pumps|  236.89137927582252|
|        Valves|  236.34508750495314|
|   Controllers|  235.15370433333288|
+--------------+--------------------+



In [127]:
df.groupBy("product_family").agg(
    F.avg("energy_kwh").alias("average_energy_usage")
).orderBy(
    F.col("average_energy_usage").desc()
).show()

+--------------+--------------------+
|product_family|average_energy_usage|
+--------------+--------------------+
|        Motors|  237.21818224597516|
|         Pumps|  236.89137927582252|
|        Valves|  236.34508750495314|
|   Controllers|  235.15370433333288|
+--------------+--------------------+



In [128]:
df.groupBy("quality_status").agg(
    F.count("*").alias("production_count")
).orderBy(
    F.col("production_count").desc()
).show()

+--------------+----------------+
|quality_status|production_count|
+--------------+----------------+
|          Pass|           74328|
|        Rework|           22928|
|        Reject|           22744|
+--------------+----------------+



In [129]:
df.groupBy("plant_id", "shift").agg(
    F.sum("units_produced").alias("total_production")
).orderBy(
    F.col("plant_id").asc(),
    F.col("total_production").desc()
).show()

+--------+-------+----------------+
|plant_id|  shift|total_production|
+--------+-------+----------------+
|    PL01|  Night|         4562612|
|    PL01|Evening|         4477769|
|    PL01|Morning|         4447851|
|    PL02|Morning|         4561068|
|    PL02|  Night|         4558467|
|    PL02|Evening|         4426917|
|    PL03|  Night|         4599608|
|    PL03|Evening|         4487211|
|    PL03|Morning|         4460950|
|    PL04|Evening|         4540617|
|    PL04|Morning|         4497099|
|    PL04|  Night|         4492219|
|    PL05|Evening|         4475101|
|    PL05|Morning|         4431446|
|    PL05|  Night|         4416616|
|    PL06|  Night|         4604432|
|    PL06|Evening|         4458798|
|    PL06|Morning|         4429107|
+--------+-------+----------------+



In [130]:
df.groupBy("plant_id", "product_family").agg(
    F.sum("defective_units").alias("total_defects")
).orderBy(
    F.col("total_defects").desc()
).show()

+--------+--------------+-------------+
|plant_id|product_family|total_defects|
+--------+--------------+-------------+
|    PL02|   Controllers|        87120|
|    PL01|         Pumps|        87103|
|    PL04|        Valves|        87084|
|    PL06|   Controllers|        86140|
|    PL06|        Valves|        85570|
|    PL03|        Valves|        85333|
|    PL03|        Motors|        84926|
|    PL01|        Valves|        84874|
|    PL03|         Pumps|        84803|
|    PL02|        Motors|        84560|
|    PL02|         Pumps|        84491|
|    PL05|        Motors|        84331|
|    PL03|   Controllers|        84202|
|    PL01|        Motors|        84174|
|    PL04|         Pumps|        83976|
|    PL04|        Motors|        83897|
|    PL06|         Pumps|        83884|
|    PL05|         Pumps|        83607|
|    PL05|   Controllers|        83589|
|    PL01|   Controllers|        83569|
+--------+--------------+-------------+
only showing top 20 rows


In [132]:
df.groupBy("machine_id").agg(
    F.avg("temperature_c").alias("average_temperature")
).orderBy(
    F.col("average_temperature").desc()
).show()

+----------+-------------------+
|machine_id|average_temperature|
+----------+-------------------+
|     MC019| 63.266223444138944|
|     MC020|  63.22489208633087|
|     MC029|  63.13119731057668|
|     MC009|   63.1252779855396|
|     MC014| 63.114309957574186|
|     MC023| 63.109051746112684|
|     MC027| 63.099668282725126|
|     MC022|  63.09403431245128|
|     MC015|  63.09026488395557|
|     MC001|   63.0778733146782|
|     MC017|  63.07674442190667|
|     MC011| 63.060230414746606|
|     MC007|    63.037661716587|
|     MC016|  63.03639870582378|
|     MC026|  63.02672521957329|
|     MC021|  63.01464069491981|
|     MC005| 62.979236446933044|
|     MC028|  62.97779683034558|
|     MC030|  62.96733774497327|
|     MC024|  62.96457946271644|
+----------+-------------------+
only showing top 20 rows


In [133]:
df.groupBy("supervisor").agg(
    F.sum("units_produced").alias("total_output")
).orderBy(
    F.col("total_output").desc()
).show()

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



**Joining DataFrames**

In [134]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

plant_df = spark.read.csv(
    "/content/plant_master.csv",
    header=True,
    inferSchema=True
)

machine_df = spark.read.csv(
    "/content/machine_master.csv",
    header=True,
    inferSchema=True
)

In [135]:
print("Production columns:", df.columns)
print("Plant columns:", plant_df.columns)
print("Machine columns:", machine_df.columns)

Production columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']
Plant columns: ['plant_id', 'plant_name', 'city', 'region', 'year_opened', 'plant_manager']
Machine columns: ['machine_id', 'plant_id', 'machine_type', 'capacity_per_hour', 'installed_year']


In [136]:
production_plant = df.join(
    plant_df,
    on="plant_id",
    how="inner"
)

production_plant.show(10, truncate=False)

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |city     |region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|PL03    |PR0000001    |2026-08-07     |MC011     |OP0275     |Evening|MTR-100     |Moto

In [137]:
production_plant.select(
    "production_id",
    "plant_id",
    "plant_name",
    "units_produced"
).show(10, truncate=False)

+-------------+--------+---------------+--------------+
|production_id|plant_id|plant_name     |units_produced|
+-------------+--------+---------------+--------------+
|PR0000001    |PL03    |Chennai Plant  |993           |
|PR0000002    |PL04    |Noida Plant    |836           |
|PR0000003    |PL01    |Hyderabad Plant|380           |
|PR0000004    |PL03    |Chennai Plant  |720           |
|PR0000005    |PL04    |Noida Plant    |296           |
|PR0000006    |PL01    |Hyderabad Plant|795           |
|PR0000007    |PL02    |Pune Plant     |955           |
|PR0000008    |PL02    |Pune Plant     |895           |
|PR0000009    |PL03    |Chennai Plant  |289           |
|PR0000010    |PL02    |Pune Plant     |527           |
+-------------+--------+---------------+--------------+
only showing top 10 rows


In [138]:
production_machine = df.join(
    machine_df,
    on="machine_id",
    how="inner"
)

production_machine.show(10, truncate=False)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|MC011     |PR0000001    |2026-08-07     |PL03    |OP0275     |Evening|MTR-100     |Motors        |1

In [139]:
p = df.alias("p")
pl = plant_df.alias("pl")
m = machine_df.alias("m")

production_full = (
    p.join(
        pl,
        F.col("p.plant_id") == F.col("pl.plant_id"),
        "left"
    )
    .join(
        m,
        F.col("p.machine_id") == F.col("m.machine_id"),
        "left"
    )
    .select(
        "p.*",
        F.col("pl.plant_name").alias("plant_name"),
        F.col("pl.city").alias("plant_city"),
        F.col("m.machine_type").alias("machine_type")
    )
)

production_full.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+----------+------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |plant_city|machine_type|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+----------+------------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |3

In [140]:
production_full.groupBy("plant_city").agg(
    F.sum("units_produced").alias("total_production")
).orderBy(
    F.col("total_production").desc()
).show()

+----------+----------------+
|plant_city|total_production|
+----------+----------------+
|   Chennai|        13547769|
|      Pune|        13546452|
|     Noida|        13529935|
|   Kolkata|        13492337|
| Hyderabad|        13488232|
| Ahmedabad|        13323163|
+----------+----------------+



In [141]:
production_full.groupBy("machine_type").agg(
    F.avg("downtime_minutes").alias("average_downtime")
).orderBy(
    F.col("average_downtime").desc()
).show()

+------------+-----------------+
|machine_type| average_downtime|
+------------+-----------------+
|     Casting| 34.1660783396419|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Testing|33.87585594724829|
|         CNC|33.64548129466162|
+------------+-----------------+



In [142]:
print(plant_df.columns)

['plant_id', 'plant_name', 'city', 'region', 'year_opened', 'plant_manager']


In [147]:
production_region = df.join(
    plant_df,
    on="plant_id",
    how="inner"
)

production_region.groupBy("region").agg(
    F.sum("defective_units").alias("total_defects")
).orderBy(
    F.col("total_defects").desc()
).show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  West|       674149|
|  East|       338993|
| North|       338118|
+------+-------------+



In [ ]:
unmatched_machines = df.join(
    machine_df.select("machine_id").distinct(),
    on="machine_id",
    how="left_anti"
)

unmatched_machines.show(20, truncate=False)

print("Unmatched records:", unmatched_machines.count())

In [145]:
p = df.alias("p")
m = machine_df.alias("m")

left_joined = p.join(
    m,
    F.col("p.machine_id") == F.col("m.machine_id"),
    "left"
)

left_joined.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|machine_id|plant_id|machine_type|capacity_per_hour|installed_year|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Even

In [148]:
unmatched_left = left_joined.filter(
    F.col("m.machine_id").isNull()
)

unmatched_left.show(20, truncate=False)

+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|machine_id|plant_id|machine_type|capacity_per_hour|installed_year|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
+-------------+---------------+--------+----------+-----------+-----+----

In [149]:
production_plant.groupBy("plant_name").agg(
    F.sum("energy_kwh").alias("total_energy_consumption")
).orderBy(
    F.col("total_energy_consumption").desc()
).show()

+---------------+------------------------+
|     plant_name|total_energy_consumption|
+---------------+------------------------+
|     Pune Plant|       4757984.949999988|
|    Noida Plant|       4746238.580000002|
|  Chennai Plant|       4742748.169999995|
|Hyderabad Plant|       4725471.189999996|
|  Kolkata Plant|       4723180.160000009|
|Ahmedabad Plant|        4672391.03000001|
+---------------+------------------------+



 **Updating Data**

In [152]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Change Reject to Failed
df = df.withColumn(
    "quality_status",
    F.when(
        F.col("quality_status") == "Reject",
        "Failed"
    ).otherwise(F.col("quality_status"))
)

# 2. Change Rework to Requires Review
df = df.withColumn(
    "quality_status",
    F.when(
        F.col("quality_status") == "Rework",
        "Requires Review"
    ).otherwise(F.col("quality_status"))
)

# 3. Increase units planned by 5% for Motors
df = df.withColumn(
    "units_planned",
    F.when(
        F.col("product_family") == "Motors",
        F.col("units_planned") * 1.05
    ).otherwise(F.col("units_planned"))
)

# 4. Increase energy consumption by 3% for Night shifts
df = df.withColumn(
    "energy_kwh",
    F.when(
        F.col("shift") == "Night",
        F.col("energy_kwh") * 1.03
    ).otherwise(F.col("energy_kwh"))
)

# 5. Replace missing operator IDs
df = df.withColumn(
    "operator_id",
    F.coalesce(F.col("operator_id"), F.lit("UNASSIGNED"))
)

# 6. Replace Anita with Anita Rao
df = df.withColumn(
    "supervisor",
    F.when(
        F.col("supervisor") == "Anita",
        "Anita Rao"
    ).otherwise(F.col("supervisor"))
)

# 7. Create corrected temperature column using the correct column 'temperature_c'
df = df.withColumn(
    "corrected_temperature",
    F.coalesce(
        F.col("temperature_c"),
        F.avg("temperature_c").over(Window.partitionBy())
    )
)

df.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned     |units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status |maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1168.65           |993           |30             |34              |388.11    |61.4

Null Values and
na Functions

In [153]:
from pyspark.sql import functions as F

In [154]:
df.filter(F.col("operator_id").isNull()).count()

0

In [160]:
print("Missing temperature:", df.filter(F.col("temperature_c").isNull()).count())

Missing temperature: 2132


In [161]:
print("Missing vibration:", df.filter(F.col("vibration_mm_s").isNull()).count())

Missing vibration: 1788


In [162]:
print("Missing downtime:", df.filter(F.col("downtime_minutes").isNull()).count())

Missing downtime: 1177


In [163]:
df_no_nulls = df.na.drop()

df_no_nulls.show(5)
print("Records after dropping nulls:", df_no_nulls.count())

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|           1168.65|           993|             30|              34|    388.11|    

In [167]:
df_temp_clean = df.na.drop(subset=["temperature_c"])

df_temp_clean.show(5)
print("Records after removing missing temperature:", df_temp_clean.count())

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|           1168.65|           993|             30|              34|    388.11|    

In [168]:
df_downtime_filled = df.na.fill(0, subset=["downtime_minutes"])

df_downtime_filled.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|           1168.65|           993|             30|              34|    388.11|    

In [169]:
df_operator_filled = df.na.fill("UNKNOWN", subset=["operator_id"])

df_operator_filled.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|     units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s| quality_status|maintenance_flag|supervisor|corrected_temperature|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|           1168.65|           993|             30|              34|    388.11|    

In [171]:
avg_temp = df.select(F.avg("temperature_c")).first()[0]

print("Average temperature:", avg_temp)

Average temperature: 63.021532392167266


In [173]:
before_count = df.count()
after_count = df.na.drop().count()

print("Records before cleansing:", before_count)
print("Records after cleansing:", after_count)
print("Records removed:", before_count - after_count)

Records before cleansing: 120000
Records after cleansing: 114985
Records removed: 5015


**Built-in Functions**

In [174]:
from pyspark.sql import functions as F


df_base = spark.read.csv(
    "/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

exercises_df = df_base.select(

    F.upper(F.col("product_family")).alias("product_family_upper"),

    F.lower(F.col("supervisor")).alias("supervisor_lower"),


    F.trim(F.col("operator_id")).alias("operator_id_trimmed"),


    F.length(F.col("product_code")).alias("product_code_length"),


    F.concat_ws("-", F.col("plant_id"), F.col("machine_id")).alias("plant_machine_combined"),


    F.round(F.col("temperature_c"), 1).alias("temperature_rounded"),


    F.abs(F.col("units_planned") - F.col("units_produced")).alias("planned_vs_produced_diff"),


    F.regexp_replace(F.col("production_id"), "^PR", "PROD_").alias("production_id_custom"),

    F.substring(F.col("product_code"), 1, 3).alias("product_code_prefix"),


    F.coalesce(F.col("temperature_c"), F.lit(0.0)).alias("temperature_coalesced")
)

exercises_df.show(10, truncate=False)

+--------------------+----------------+-------------------+-------------------+----------------------+-------------------+------------------------+--------------------+-------------------+---------------------+
|product_family_upper|supervisor_lower|operator_id_trimmed|product_code_length|plant_machine_combined|temperature_rounded|planned_vs_produced_diff|production_id_custom|product_code_prefix|temperature_coalesced|
+--------------------+----------------+-------------------+-------------------+----------------------+-------------------+------------------------+--------------------+-------------------+---------------------+
|MOTORS              |charan          |OP0275             |7                  |PL03-MC011            |61.4               |67                      |PROD_0000001        |MTR                |61.4                 |
|CONTROLLERS         |farah           |OP0489             |7                  |PL04-MC019            |64.4               |27                      |PROD_0000

when()
 /
otherwise()

In [175]:
from pyspark.sql import functions as F

In [176]:
df = df.withColumn(
    "efficiency_percentage",
    F.when(
        F.col("units_planned") != 0,
        (F.col("units_produced") / F.col("units_planned")) * 100
    )
)

df = df.withColumn(
    "efficiency_category",
    F.when(F.col("efficiency_percentage") >= 100, "Above Target")
     .when(F.col("efficiency_percentage") >= 95, "On Target")
     .when(F.col("efficiency_percentage") >= 85, "Below Target")
     .otherwise("Critical")
)

df.select(
    "units_produced",
    "units_planned",
    "efficiency_percentage",
    "efficiency_category"
).show(10)

+--------------+------------------+---------------------+-------------------+
|units_produced|     units_planned|efficiency_percentage|efficiency_category|
+--------------+------------------+---------------------+-------------------+
|           993|           1168.65|    84.96983699140033|           Critical|
|           836|             809.0|   103.33745364647713|       Above Target|
|           380|487.30500000000006|    77.97990991268301|           Critical|
|           720|             787.0|     91.4866581956798|       Below Target|
|           296|             344.0|    86.04651162790698|       Below Target|
|           795|1056.1950000000002|    75.27019158394044|           Critical|
|           955|            1165.0|    81.97424892703863|           Critical|
|           895|             862.0|   103.82830626450117|       Above Target|
|           289|             313.0|    92.33226837060703|       Below Target|
|           527|             532.0|    99.06015037593986|       

In [177]:
df = df.withColumn(
    "defect_category",
    F.when(F.col("defective_units") == 0, "Perfect")
     .when(F.col("defective_units") <= 10, "Normal")
     .when(F.col("defective_units") <= 25, "Attention")
     .otherwise("High Defect")
)

df.select(
    "defective_units",
    "defect_category"
).show(10)

+---------------+---------------+
|defective_units|defect_category|
+---------------+---------------+
|             30|    High Defect|
|              0|        Perfect|
|             17|      Attention|
|              7|         Normal|
|              7|         Normal|
|             50|    High Defect|
|             10|         Normal|
|             51|    High Defect|
|              8|         Normal|
|             26|    High Defect|
+---------------+---------------+
only showing top 10 rows


In [179]:
df = df.withColumn(
    "machine_condition",
    F.when(
        (F.col("temperature_c") > 80) |
        (F.col("vibration_mm_s") > 10),
        "Inspect"
    ).otherwise("Normal")
)

df.select(
    "temperature_c",
    "vibration_mm_s",
    "machine_condition"
).show(10)

+-------------+--------------+-----------------+
|temperature_c|vibration_mm_s|machine_condition|
+-------------+--------------+-----------------+
|         61.4|          3.88|           Normal|
|        64.37|          2.28|           Normal|
|        54.13|          4.66|           Normal|
|        64.47|          NULL|           Normal|
|        72.59|          3.21|           Normal|
|        64.91|          2.77|           Normal|
|        62.76|          2.89|           Normal|
|         57.1|          1.34|           Normal|
|        66.55|          3.07|           Normal|
|        60.96|          3.48|           Normal|
+-------------+--------------+-----------------+
only showing top 10 rows


In [180]:
df.groupBy("efficiency_category") \
  .count() \
  .orderBy("efficiency_category") \
  .show()

+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|       Above Target|16354|
|       Below Target|54747|
|           Critical|28651|
|          On Target|20248|
+-------------------+-----+



In [181]:
df.groupBy("defect_category") \
  .count() \
  .orderBy("defect_category") \
  .show()

+---------------+-----+
|defect_category|count|
+---------------+-----+
|      Attention|45745|
|    High Defect|28058|
|         Normal|34706|
|        Perfect|11491|
+---------------+-----+



In [182]:
df.groupBy("machine_condition") \
  .count() \
  .show()

+-----------------+------+
|machine_condition| count|
+-----------------+------+
|          Inspect|  1394|
|           Normal|118606|
+-----------------+------+



In [184]:
df.select(
    "production_id",
    "efficiency_percentage",
    "efficiency_category",
    "defective_units",
    "defect_category",
    "temperature_c",
    "vibration_mm_s",
    "machine_condition"
).show(20, truncate=False)

+-------------+---------------------+-------------------+---------------+---------------+-------------+--------------+-----------------+
|production_id|efficiency_percentage|efficiency_category|defective_units|defect_category|temperature_c|vibration_mm_s|machine_condition|
+-------------+---------------------+-------------------+---------------+---------------+-------------+--------------+-----------------+
|PR0000001    |84.96983699140033    |Critical           |30             |High Defect    |61.4         |3.88          |Normal           |
|PR0000002    |103.33745364647713   |Above Target       |0              |Perfect        |64.37        |2.28          |Normal           |
|PR0000003    |77.97990991268301    |Critical           |17             |Attention      |54.13        |4.66          |Normal           |
|PR0000004    |91.4866581956798     |Below Target       |7              |Normal         |64.47        |NULL          |Normal           |
|PR0000005    |86.04651162790698    |Belo

**Complex JSON**

In [185]:
from pyspark.sql import functions as F

machine_json = spark.read.option(
    "multiline", "true"
).json("/content/machine_configurations.json")

In [186]:
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



In [187]:
machine_json.select(
    "machine_id",
    "machine_type"
).show(truncate=False)

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|MC001     |CNC         |
|MC002     |Assembly    |
|MC003     |Casting     |
|MC004     |Welding     |
|MC005     |Testing     |
|MC006     |CNC         |
|MC007     |Assembly    |
|MC008     |Casting     |
|MC009     |Welding     |
|MC010     |Testing     |
|MC011     |CNC         |
|MC012     |Assembly    |
|MC013     |Casting     |
|MC014     |Welding     |
|MC015     |Testing     |
|MC016     |CNC         |
|MC017     |Assembly    |
|MC018     |Casting     |
|MC019     |Welding     |
|MC020     |Testing     |
+----------+------------+
only showing top 20 rows


In [188]:
machine_json.select(
    "machine_id",
    F.col("specifications.rated_power_kw").alias("rated_power_kw")
).show(truncate=False)

+----------+--------------+
|machine_id|rated_power_kw|
+----------+--------------+
|MC001     |22            |
|MC002     |22            |
|MC003     |55            |
|MC004     |22            |
|MC005     |30            |
|MC006     |30            |
|MC007     |22            |
|MC008     |55            |
|MC009     |30            |
|MC010     |55            |
|MC011     |30            |
|MC012     |55            |
|MC013     |30            |
|MC014     |30            |
|MC015     |45            |
|MC016     |30            |
|MC017     |30            |
|MC018     |30            |
|MC019     |30            |
|MC020     |15            |
+----------+--------------+
only showing top 20 rows


In [190]:
machine_json.select(
    "machine_id",
    F.col("specifications.max_temperature_c").alias("max_temperature_c")
).show(truncate=False)

+----------+-----------------+
|machine_id|max_temperature_c|
+----------+-----------------+
|MC001     |85               |
|MC002     |85               |
|MC003     |90               |
|MC004     |85               |
|MC005     |85               |
|MC006     |85               |
|MC007     |80               |
|MC008     |75               |
|MC009     |85               |
|MC010     |90               |
|MC011     |90               |
|MC012     |80               |
|MC013     |90               |
|MC014     |80               |
|MC015     |90               |
|MC016     |75               |
|MC017     |85               |
|MC018     |85               |
|MC019     |75               |
|MC020     |80               |
+----------+-----------------+
only showing top 20 rows


In [194]:
machine_json.select(
    "machine_id",
    F.col("specifications.max_temperature_c").alias("maximum_temperature")
).show(truncate=False)

+----------+-------------------+
|machine_id|maximum_temperature|
+----------+-------------------+
|MC001     |85                 |
|MC002     |85                 |
|MC003     |90                 |
|MC004     |85                 |
|MC005     |85                 |
|MC006     |85                 |
|MC007     |80                 |
|MC008     |75                 |
|MC009     |85                 |
|MC010     |90                 |
|MC011     |90                 |
|MC012     |80                 |
|MC013     |90                 |
|MC014     |80                 |
|MC015     |90                 |
|MC016     |75                 |
|MC017     |85                 |
|MC018     |85                 |
|MC019     |75                 |
|MC020     |80                 |
+----------+-------------------+
only showing top 20 rows


In [197]:
machine_json.select(
    "machine_id",
    F.col("specifications.vibration_limit_mm_s").alias("vibration_limit")
).show(truncate=False)

+----------+---------------+
|machine_id|vibration_limit|
+----------+---------------+
|MC001     |5.0            |
|MC002     |5.5            |
|MC003     |4.5            |
|MC004     |4.5            |
|MC005     |4.5            |
|MC006     |5.5            |
|MC007     |4.5            |
|MC008     |5.0            |
|MC009     |5.5            |
|MC010     |4.5            |
|MC011     |6.0            |
|MC012     |5.5            |
|MC013     |5.0            |
|MC014     |5.5            |
|MC015     |4.5            |
|MC016     |5.5            |
|MC017     |4.5            |
|MC018     |5.0            |
|MC019     |5.0            |
|MC020     |5.0            |
+----------+---------------+
only showing top 20 rows


In [203]:
import pyspark.sql.functions as F

machine_json.select(
    "machine_id",
    F.col("maintenance.frequency_days").alias("maintenance_frequency_days")
).show(truncate=False)

+----------+--------------------------+
|machine_id|maintenance_frequency_days|
+----------+--------------------------+
|MC001     |60                        |
|MC002     |60                        |
|MC003     |45                        |
|MC004     |60                        |
|MC005     |30                        |
|MC006     |60                        |
|MC007     |30                        |
|MC008     |30                        |
|MC009     |30                        |
|MC010     |45                        |
|MC011     |30                        |
|MC012     |30                        |
|MC013     |60                        |
|MC014     |90                        |
|MC015     |30                        |
|MC016     |45                        |
|MC017     |30                        |
|MC018     |30                        |
|MC019     |30                        |
|MC020     |45                        |
+----------+--------------------------+
only showing top 20 rows


In [200]:
sensor_df = machine_json.select(
    "machine_id",
    F.explode("sensors").alias("sensor")
)

sensor_df.show(truncate=False)

+----------+-----------------------+
|machine_id|sensor                 |
+----------+-----------------------+
|MC001     |{true, temperature, C} |
|MC001     |{true, vibration, mm/s}|
|MC001     |{true, energy, kWh}    |
|MC002     |{true, temperature, C} |
|MC002     |{true, vibration, mm/s}|
|MC002     |{true, energy, kWh}    |
|MC003     |{true, temperature, C} |
|MC003     |{true, vibration, mm/s}|
|MC003     |{true, energy, kWh}    |
|MC004     |{true, temperature, C} |
|MC004     |{true, vibration, mm/s}|
|MC004     |{true, energy, kWh}    |
|MC005     |{true, temperature, C} |
|MC005     |{true, vibration, mm/s}|
|MC005     |{true, energy, kWh}    |
|MC006     |{true, temperature, C} |
|MC006     |{true, vibration, mm/s}|
|MC006     |{true, energy, kWh}    |
|MC007     |{true, temperature, C} |
|MC007     |{true, vibration, mm/s}|
+----------+-----------------------+
only showing top 20 rows


In [204]:
sensor_details = sensor_df.select(
    "machine_id",
    F.col("sensor.sensor").alias("sensor_type"),
    F.col("sensor.unit").alias("sensor_unit"),
    F.col("sensor.active").alias("active")
)

sensor_details.show(truncate=False)

+----------+-----------+-----------+------+
|machine_id|sensor_type|sensor_unit|active|
+----------+-----------+-----------+------+
|MC001     |temperature|C          |true  |
|MC001     |vibration  |mm/s       |true  |
|MC001     |energy     |kWh        |true  |
|MC002     |temperature|C          |true  |
|MC002     |vibration  |mm/s       |true  |
|MC002     |energy     |kWh        |true  |
|MC003     |temperature|C          |true  |
|MC003     |vibration  |mm/s       |true  |
|MC003     |energy     |kWh        |true  |
|MC004     |temperature|C          |true  |
|MC004     |vibration  |mm/s       |true  |
|MC004     |energy     |kWh        |true  |
|MC005     |temperature|C          |true  |
|MC005     |vibration  |mm/s       |true  |
|MC005     |energy     |kWh        |true  |
|MC006     |temperature|C          |true  |
|MC006     |vibration  |mm/s       |true  |
|MC006     |energy     |kWh        |true  |
|MC007     |temperature|C          |true  |
|MC007     |vibration  |mm/s    

In [206]:
active_temperature_machines = sensor_details.filter(
    (F.lower(F.col("sensor_type")) == "temperature") &
    (F.col("active") == True)
)

active_temperature_machines.show(truncate=False)

+----------+-----------+-----------+------+
|machine_id|sensor_type|sensor_unit|active|
+----------+-----------+-----------+------+
|MC001     |temperature|C          |true  |
|MC002     |temperature|C          |true  |
|MC003     |temperature|C          |true  |
|MC004     |temperature|C          |true  |
|MC005     |temperature|C          |true  |
|MC006     |temperature|C          |true  |
|MC007     |temperature|C          |true  |
|MC008     |temperature|C          |true  |
|MC009     |temperature|C          |true  |
|MC010     |temperature|C          |true  |
|MC011     |temperature|C          |true  |
|MC012     |temperature|C          |true  |
|MC013     |temperature|C          |true  |
|MC014     |temperature|C          |true  |
|MC015     |temperature|C          |true  |
|MC016     |temperature|C          |true  |
|MC017     |temperature|C          |true  |
|MC018     |temperature|C          |true  |
|MC019     |temperature|C          |true  |
|MC020     |temperature|C       

In [205]:
production_with_config = df.join(
    machine_json,
    on="machine_id",
    how="left"
)

production_with_config.show(10, truncate=False)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+------------------+--------------+---------------+----------------+----------+-------------+--------------+---------------+----------------+----------+---------------------+---------------------+-------------------+---------------+-----------------+-----------------+--------------+------------+----------------+--------+----------------------------------------------------------------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift  |product_code|product_family|units_planned     |units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status |maintenance_flag|supervisor|corrected_temperature|efficiency_percentage|efficiency_category|defect_category|machine_condition|capacity_per_hour|installed_year|machine_type|maintenance     |plant_id|sensors                                                               |sp

**Union and Union ALL**

In [220]:
from pyspark.sql import functions as F


extra_df = spark.read.csv(
    "/content/manufacturing_production_extra_12k.csv",
    header=True,
    inferSchema=True
)

print("1. Main DataFrame count:", df.count())


print("2. Extra DataFrame count:", extra_df.count())

print("\nMain DataFrame columns:", df.columns)
print("Extra DataFrame columns:", extra_df.columns)


if df.columns == extra_df.columns:
    union_df = df.union(extra_df)
    print("\n3. Union count:", union_df.count())
else:
    print("\n3. union() needs matching column names in the same order.")
    union_df = None


if df.columns == extra_df.columns:
    union_all_df = df.unionAll(extra_df)
    print("4. UnionAll count:", union_all_df.count())
else:
    print("4. unionAll() needs matching column names in the same order.")
    union_all_df = None


if union_df is not None and union_all_df is not None:
    print("\n5. union() count:", union_df.count())
    print("   unionAll() count:", union_all_df.count())

combined_df = df.unionByName(extra_df, allowMissingColumns=True)

duplicate_ids = (
    combined_df
    .groupBy("production_id")
    .count()
    .filter(F.col("count") > 1)
)

print("\n6. Duplicate production_id values:")
duplicate_ids.show(truncate=False)


deduplicated_df = combined_df.dropDuplicates()

print("\n7. Data after removing duplicate rows:")
deduplicated_df.show(5, truncate=False)


print("\n8. Count before dropDuplicates():", combined_df.count())
print("   Count after dropDuplicates():", deduplicated_df.count())


union_by_name_df = df.unionByName(
    extra_df,
    allowMissingColumns=True
)

print("\n9. unionByName() count:", union_by_name_df.count())
union_by_name_df.show(5, truncate=False)


if set(df.columns) == set(extra_df.columns):
    reordered_extra_df = extra_df.select(*reversed(extra_df.columns))

    rearranged_union_df = df.unionByName(reordered_extra_df)

    print("\n10. unionByName() after rearranging columns:")
    print("Count:", rearranged_union_df.count())
    rearranged_union_df.show(5, truncate=False)
else:
    print("\n10. The DataFrames have different columns.")
    print("unionByName() can align shared columns when allowMissingColumns=True.")

1. Main DataFrame count: 120000
2. Extra DataFrame count: 12500

Main DataFrame columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor', 'corrected_temperature', 'efficiency_percentage', 'efficiency_category', 'defect_category', 'machine_condition', 'year', 'month', 'day', 'month_name', 'day_name', 'year_month', 'date_plus_7_days', 'date_minus_30_days', 'days_from_current_date']
Extra DataFrame columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']

3. union() needs matching column names in the sa

**Window Functions**


In [218]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


plant_window = Window.partitionBy("plant_id").orderBy(
    F.col("units_produced").desc()
)


ranked_df = (
    df.withColumn("row_number", F.row_number().over(plant_window))
      .withColumn("rank", F.rank().over(plant_window))
      .withColumn("dense_rank", F.dense_rank().over(plant_window))
)

ranked_df.select(
    "plant_id", "production_id", "units_produced",
    "row_number", "rank", "dense_rank"
).show(20, truncate=False)


top5_production = ranked_df.filter(F.col("row_number") <= 5)
top5_production.show(truncate=False)


machine_output = (
    df.groupBy("plant_id", "machine_id")
      .agg(F.sum("units_produced").alias("total_output"))
)

machine_rank_window = Window.partitionBy("plant_id").orderBy(
    F.col("total_output").desc()
)

top3_machines = (
    machine_output
    .withColumn("machine_rank", F.dense_rank().over(machine_rank_window))
    .filter(F.col("machine_rank") <= 3)
)

top3_machines.show(truncate=False)


plant_total_window = Window.partitionBy("plant_id")

plant_totals = df.withColumn(
    "total_plant_production",
    F.sum("units_produced").over(plant_total_window)
)

plant_totals.select(
    "plant_id", "production_id", "units_produced",
    "total_plant_production"
).show(10, truncate=False)


machine_avg_window = Window.partitionBy("machine_id")

machine_avg_df = df.withColumn(
    "average_machine_output",
    F.avg("units_produced").over(machine_avg_window)
)

machine_avg_df.select(
    "machine_id", "production_id", "units_produced",
    "average_machine_output"
).show(10, truncate=False)


cumulative_window = (
    Window.partitionBy("machine_id")
    .orderBy("production_date", "production_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

cumulative_df = df.withColumn(
    "cumulative_production",
    F.sum("units_produced").over(cumulative_window)
)

cumulative_df.select(
    "machine_id", "production_date", "units_produced",
    "cumulative_production"
).show(20, truncate=False)


machine_window = Window.partitionBy("machine_id").orderBy(
    "production_date", "production_id"
)

previous_df = df.withColumn(
    "previous_output",
    F.lag("units_produced", 1).over(machine_window)
)

previous_df.select(
    "machine_id", "production_date", "units_produced",
    "previous_output"
).show(20, truncate=False)


next_df = df.withColumn(
    "next_output",
    F.lead("units_produced", 1).over(machine_window)
)

next_df.select(
    "machine_id", "production_date", "units_produced",
    "next_output"
).show(20, truncate=False)


change_df = (
    df.withColumn(
        "previous_output",
        F.lag("units_produced", 1).over(machine_window)
    )
    .withColumn(
        "production_change",
        F.col("units_produced") - F.col("previous_output")
    )
)

change_df.select(
    "machine_id", "production_date", "units_produced",
    "previous_output", "production_change"
).show(20, truncate=False)


family_output = (
    df.groupBy("product_family")
      .agg(F.sum("units_produced").alias("total_output"))
)

family_window = Window.orderBy(F.col("total_output").desc())

family_ranked_df = family_output.withColumn(
    "family_rank",
    F.dense_rank().over(family_window)
)

family_ranked_df.show(truncate=False)

+--------+-------------+--------------+----------+----+----------+
|plant_id|production_id|units_produced|row_number|rank|dense_rank|
+--------+-------------+--------------+----------+----+----------+
|PL01    |PR0073059    |1239          |1         |1   |1         |
|PL01    |PR0002678    |1238          |2         |2   |2         |
|PL01    |PR0005192    |1236          |3         |3   |3         |
|PL01    |PR0044516    |1236          |4         |3   |3         |
|PL01    |PR0056319    |1236          |5         |3   |3         |
|PL01    |PR0006350    |1234          |6         |6   |4         |
|PL01    |PR0046906    |1233          |7         |7   |5         |
|PL01    |PR0046582    |1230          |8         |8   |6         |
|PL01    |PR0024941    |1229          |9         |9   |7         |
|PL01    |PR0088933    |1229          |10        |9   |7         |
|PL01    |PR0097172    |1229          |11        |9   |7         |
|PL01    |PR0048075    |1228          |12        |12  |8      

 **Date and Time Functions**

In [219]:
from pyspark.sql import functions as F


df = df.withColumn(
    "production_date",
    F.to_date("production_date")
)


df = df.withColumn(
    "year",
    F.year("production_date")
)


df = df.withColumn(
    "month",
    F.month("production_date")
)


df = df.withColumn(
    "day",
    F.dayofmonth("production_date")
)


df = df.withColumn(
    "month_name",
    F.date_format("production_date", "MMMM")
)


df = df.withColumn(
    "day_name",
    F.date_format("production_date", "EEEE")
)


df = df.withColumn(
    "year_month",
    F.date_format("production_date", "yyyy-MM")
)


df.select(
    "production_date",
    "year",
    "month",
    "day",
    "month_name",
    "day_name",
    "year_month"
).show(10, truncate=False)

monthly_production = (
    df.groupBy("year_month")
      .agg(F.sum("units_produced").alias("total_production"))
      .orderBy("year_month")
)

monthly_production.show(truncate=False)


monthly_defects = (
    df.groupBy("year_month")
      .agg(F.sum("defective_units").alias("total_defects"))
      .orderBy("year_month")
)

monthly_defects.show(truncate=False)


plant_monthly_production = (
    df.groupBy("plant_id", "year_month")
      .agg(F.sum("units_produced").alias("total_production"))
      .orderBy("plant_id", "year_month")
)

plant_monthly_production.show(truncate=False)


production_between_dates = df.filter(
    F.col("production_date").between(
        F.lit("2026-01-01").cast("date"),
        F.lit("2026-12-31").cast("date")
    )
)

production_between_dates.show(10, truncate=False)


df = df.withColumn(
    "date_plus_7_days",
    F.date_add("production_date", 7)
)


df = df.withColumn(
    "date_minus_30_days",
    F.date_sub("production_date", 30)
)

df = df.withColumn(
    "days_from_current_date",
    F.datediff(F.current_date(), F.col("production_date"))
)

df.select(
    "production_date",
    "date_plus_7_days",
    "date_minus_30_days",
    "days_from_current_date"
).show(10, truncate=False)


highest_output_month = (
    df.groupBy("year_month")
      .agg(F.sum("units_produced").alias("total_output"))
      .orderBy(F.col("total_output").desc())
)

highest_output_month.show(1, truncate=False)

+---------------+----+-----+---+----------+---------+----------+
|production_date|year|month|day|month_name|day_name |year_month|
+---------------+----+-----+---+----------+---------+----------+
|2026-08-07     |2026|8    |7  |August    |Friday   |2026-08   |
|2026-08-03     |2026|8    |3  |August    |Monday   |2026-08   |
|2026-05-20     |2026|5    |20 |May       |Wednesday|2026-05   |
|2026-01-15     |2026|1    |15 |January   |Thursday |2026-01   |
|2026-09-21     |2026|9    |21 |September |Monday   |2026-09   |
|2026-09-03     |2026|9    |3  |September |Thursday |2026-09   |
|2026-05-07     |2026|5    |7  |May       |Thursday |2026-05   |
|2026-08-17     |2026|8    |17 |August    |Monday   |2026-08   |
|2026-03-10     |2026|3    |10 |March     |Tuesday  |2026-03   |
|2026-02-13     |2026|2    |13 |February  |Friday   |2026-02   |
+---------------+----+-----+---+----------+---------+----------+
only showing top 10 rows
+----------+----------------+
|year_month|total_production|
+----